In [ ]:
!pip install chromadb sentence-transformers groq langgraph langchain-core python-whois requests beautifulsoup4 dnspython

In [2]:
import os
import json
import time
import re
import requests
import whois
import dns.resolver
import chromadb
from datetime import datetime, timezone
from typing import TypedDict
from bs4 import BeautifulSoup
from groq import Groq
from google.colab import userdata
from langgraph.graph import StateGraph, END
from sentence_transformers import SentenceTransformer
from chromadb.utils import embedding_functions

os.environ["GROQ_API_KEY"]       = userdata.get("GROQ_API_KEY")
os.environ["VIRUSTOTAL_API_KEY"] = userdata.get("VIRUSTOTAL_API_KEY")

client = Groq(api_key=os.environ["GROQ_API_KEY"])
VT_KEY = os.environ["VIRUSTOTAL_API_KEY"]
MODEL  = "llama-3.3-70b-versatile"

print("✅ All imports loaded")
print("✅ Credentials ready")


✅ All imports loaded
✅ Credentials ready


In [3]:
# CONCEPT: The embedding model converts text to vectors.
# We use all-MiniLM-L6-v2 — small, fast, free, no GPU needed.
# Downloads once (~90MB) and caches locally in Colab.
#
# YOUR KNOWLEDGE BRIDGE:
# TF-IDF:              text → sparse vector (mostly zeros)
# Sentence transformer: text → dense vector (384 floats, all meaningful)
#
# Both use cosine similarity for comparison — same math you know.

print("\nLoading embedding model (downloads once, ~90MB)...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
print("✅ Embedding model loaded")
print(f"   Model produces {embedding_model.get_sentence_embedding_dimension()}-dimensional vectors")

# Quick test to verify embeddings work
test_vec = embedding_model.encode("test message")
print(f"   Test vector shape: {test_vec.shape}")
print(f"   First 5 values  : {test_vec[:5].tolist()}")



Loading embedding model (downloads once, ~90MB)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded
   Model produces 384-dimensional vectors


/tmp/ipykernel_34079/2734062462.py:14: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"   Model produces {embedding_model.get_sentence_embedding_dimension()}-dimensional vectors")


   Test vector shape: (384,)
   First 5 values  : [0.029340405017137527, 0.04244305565953255, -0.0034467156510800123, 0.061912085860967636, 0.011817693710327148]


In [4]:
# CONCEPT: ChromaDB is a local vector database.
# It stores: vectors + original text + metadata (verdict, score)
# You query it with a vector and it returns the most similar stored vectors.
#
# Think of it as a dictionary where:
#   keys   = vectors (numerical representations of past cases)
#   values = the original case data (text + verdict + evidence)
#   search = "find me keys closest to this new key"

# Initialize ChromaDB — stores data in /content/chroma_db
chroma_client = chromadb.PersistentClient(path="/content/chroma_db")

# Create or load the scam cases collection
# A collection = a table in a traditional database
# Each row = one investigated case with its vector + metadata
collection = chroma_client.get_or_create_collection(
    name="scam_cases",
    metadata={"hnsw:space": "cosine"}  # Use cosine similarity
)

print(f"\n✅ ChromaDB initialized")
print(f"   Storage path     : /content/chroma_db")
print(f"   Collection       : scam_cases")
print(f"   Cases stored     : {collection.count()}")
print(f"   Similarity metric: cosine")



✅ ChromaDB initialized
   Storage path     : /content/chroma_db
   Collection       : scam_cases
   Cases stored     : 0
   Similarity metric: cosine


In [5]:
# CONCEPT: Three operations on memory:
#   1. store_case  → save investigated case to ChromaDB
#   2. retrieve_similar → find similar past cases by cosine similarity
#   3. build_memory_context → format past cases for the LLM to read

def store_case(case_id: str,
               message: str,
               verdict: dict,
               agents_findings: dict) -> None:
    """
    Stores an investigated case in ChromaDB memory.

    CONCEPT: We store three things per case:
    1. The embedding vector  → for similarity search
    2. The original message  → to show analysts what matched
    3. Metadata (verdict etc)→ ChromaDB metadata for filtering

    YOUR KNOWLEDGE: This is like fitting a TF-IDF vectorizer
    and storing the resulting vector alongside the document.
    """
    # Convert message to embedding vector
    # CONCEPT: Same message always produces same vector (deterministic)
    vector = embedding_model.encode(message).tolist()

    # Prepare metadata — ChromaDB stores this alongside the vector
    # Metadata must be flat (no nested dicts) — ChromaDB requirement
    metadata = {
        "case_id"          : case_id,
        "is_scam"          : str(verdict.get("is_scam", False)),
        "confidence"       : int(verdict.get("confidence", 0)),
        "scam_type"        : str(verdict.get("scam_type", "unknown")),
        "timestamp"        : datetime.now(timezone.utc).isoformat(),
        "technical_risk"   : str(agents_findings.get(
            "technical_risk", "unknown")),
        "cultural_risk"    : str(agents_findings.get(
            "cultural_risk",  "unknown")),
        "nlp_risk"         : str(agents_findings.get(
            "nlp_risk",       "unknown")),
        "key_red_flags"    : json.dumps(
            verdict.get("red_flags", [])[:3]),  # store top 3 flags
    }

    # Store in ChromaDB
    # CONCEPT: upsert = insert if new, update if case_id exists
    collection.upsert(
        ids       = [case_id],
        embeddings= [vector],
        documents = [message[:500]],   # store first 500 chars of message
        metadatas = [metadata]
    )

    print(f"   💾 Stored case {case_id} in memory")
    print(f"      Scam: {verdict.get('is_scam')} | "
          f"Score: {verdict.get('confidence')} | "
          f"Type: {verdict.get('scam_type')}")


def retrieve_similar(message: str,
                     n_results: int = 3,
                     similarity_threshold: float = 0.65) -> list:
    """
    Retrieves the most similar past cases from ChromaDB.

    CONCEPT: This is cosine similarity search — exactly what you
    already know, but done at scale across all stored cases.

    similarity_threshold = 0.65 means we only return cases with
    cosine similarity >= 0.65 (reasonably similar, not just any case)

    YOUR KNOWLEDGE:
    cosine_similarity(v1, v2) = (v1 · v2) / (|v1| × |v2|)
    ChromaDB computes this for us across all stored vectors.
    1.0 = identical meaning, 0.0 = completely different
    """
    # If no cases stored yet, return empty
    if collection.count() == 0:
        return []

    # Convert new message to vector
    query_vector = embedding_model.encode(message).tolist()

    # Search ChromaDB — returns n_results closest vectors
    # CONCEPT: ChromaDB uses HNSW (Hierarchical Navigable Small World)
    # graph index for fast approximate nearest neighbor search
    results = collection.query(
        query_embeddings=[query_vector],
        n_results=min(n_results, collection.count()),
        include=["documents", "metadatas", "distances"]
    )

    # Format results — convert ChromaDB output to clean list
    similar_cases = []
    for i in range(len(results["ids"][0])):
        # ChromaDB cosine distance: 0=identical, 2=opposite
        # Convert to similarity: similarity = 1 - (distance/2)
        distance   = results["distances"][0][i]
        similarity = round(1 - (distance / 2), 3)

        # Only include cases above similarity threshold
        if similarity >= similarity_threshold:
            similar_cases.append({
                "case_id"   : results["ids"][0][i],
                "similarity": similarity,
                "message"   : results["documents"][0][i],
                "metadata"  : results["metadatas"][0][i],
            })

    return similar_cases


def build_memory_context(similar_cases: list) -> str:
    """
    Formats retrieved past cases into text the LLM can read.

    CONCEPT: This is the RAG (Retrieval Augmented Generation) step.
    We take raw vector search results and format them as readable
    context that gets added to the orchestrator's prompt.
    The LLM reads past cases as evidence, not just current signals.
    """
    if not similar_cases:
        return "No similar past cases found in memory."

    context_lines = [
        f"MEMORY: {len(similar_cases)} similar past case(s) retrieved:\n"
    ]

    for i, case in enumerate(similar_cases, 1):
        meta       = case["metadata"]
        similarity = case["similarity"]
        is_scam    = meta.get("is_scam", "unknown")
        confidence = meta.get("confidence", 0)
        scam_type  = meta.get("scam_type",  "unknown")
        flags      = json.loads(meta.get("key_red_flags", "[]"))

        context_lines.append(
            f"Past Case #{i} (similarity: {similarity:.0%}):\n"
            f"  Verdict   : {'SCAM' if is_scam=='True' else 'SAFE'} "
            f"({confidence}% confidence)\n"
            f"  Scam type : {scam_type}\n"
            f"  Red flags : {', '.join(flags) if flags else 'none'}\n"
            f"  Message   : {case['message'][:150]}...\n"
        )

    return "\n".join(context_lines)



In [6]:
# CONCEPT: Past cases can boost or reduce the current score.
# If 3 very similar past cases were all confirmed scams
# the new case gets a confidence boost — pattern is established.
# This is the "learning" component of the system.

def calculate_memory_boost(similar_cases: list,
                           current_score: int) -> dict:
    """
    Adjusts the risk score based on similar past cases.

    BOOST LOGIC:
    - High similarity (>0.85) confirmed scam → +10 points
    - Medium similarity (>0.70) confirmed scam → +5 points
    - High similarity confirmed safe → -5 points (reduce false positives)
    - Multiple consistent past scams → additional +5 points

    Maximum boost: +20 points
    Maximum reduction: -10 points
    """
    if current_score < 45:
        return {
            "boosted_score": current_score,
            "boost_applied": 0,
            "boost_reason" : ["Base score too low for memory boost "
                              "— agents found insufficient signals"]
        }
    if not similar_cases:
        return {
            "boosted_score": current_score,
            "boost_applied": 0,
            "boost_reason" : "No similar past cases — no memory adjustment"
        }

    boost = 0
    reasons = []

    confirmed_scams = [c for c in similar_cases
                       if c["metadata"].get("is_scam") == "True"]
    confirmed_safe  = [c for c in similar_cases
                       if c["metadata"].get("is_scam") == "False"]

    # High similarity scam matches
    for case in confirmed_scams:
        sim = case["similarity"]
        if sim >= 0.90:
            boost += 10
            reasons.append(
                f"Very similar past scam found (similarity: {sim:.0%})")
        elif sim >= 0.80:
            boost += 5
            reasons.append(
                f"Similar past scam found (similarity: {sim:.0%})")

    # Multiple consistent scam matches — pattern established
    if len(confirmed_scams) >= 2:
        boost += 5
        reasons.append(
            f"{len(confirmed_scams)} past scam cases with similar pattern")

    # Safe case matches — reduce false positives
    for case in confirmed_safe:
        if case["similarity"] >= 0.85:
            boost -= 5
            reasons.append(
                f"Very similar past SAFE case (similarity: "
                f"{case['similarity']:.0%})")

    # Cap boost range
    boost = max(-10, min(20, boost))

    boosted_score = min(100, max(0, current_score + boost))

    return {
        "boosted_score"     : boosted_score,
        "original_score"    : current_score,
        "boost_applied"     : boost,
        "boost_reason"      : reasons,
        "similar_scams"     : len(confirmed_scams),
        "similar_safe"      : len(confirmed_safe),
        "total_cases_found" : len(similar_cases)
    }


In [7]:
# CONCEPT: Before we test, we seed ChromaDB with known scam
# and legitimate cases. This gives the system prior knowledge
# to learn from immediately — like a fraud analyst reading
# past case files on their first day.
#
# In production this would grow automatically with every case.
# We seed manually here to demonstrate the memory boost effect.

seed_cases = [
    {
        "id"     : "seed_001",
        "message": """From: hralert@wadialsagroup.com
            Dear Candidate, You have been selected for an interview
            at ADNOC contractor Wadi Al Salam Group. Salary AED 18,000
            per month. Please bring Emirates ID and pay AED 300
            processing fee. HR Department""",
        "verdict": {
            "is_scam"   : True,
            "confidence": 80,
            "scam_type" : "job_scam",
            "red_flags" : ["ADNOC impersonation",
                           "processing fee requested",
                           "unrealistic salary"]
        },
        "risks": {
            "technical_risk": "HIGH",
            "cultural_risk" : "HIGH",
            "nlp_risk"      : "MEDIUM"
        }
    },
    {
        "id"     : "seed_002",
        "message": """URGENT: Your Emirates NBD account has been suspended.
            Click to verify immediately: http://emiratesnbd-verify.tk
            Failure to verify in 2 hours will permanently close account.""",
        "verdict": {
            "is_scam"   : True,
            "confidence": 95,
            "scam_type" : "phishing",
            "red_flags" : ["suspicious URL", "urgency threat",
                           "account suspension threat"]
        },
        "risks": {
            "technical_risk": "VERY HIGH",
            "cultural_risk" : "HIGH",
            "nlp_risk"      : "HIGH"
        }
    },
    {
        "id"     : "seed_003",
        "message": """Congratulations! You have won AED 500,000 in the
            Dubai Government Lucky Draw. Send your Emirates ID and
            pay AED 250 processing fee to claim your prize.
            Offer expires in 24 hours.""",
        "verdict": {
            "is_scam"   : True,
            "confidence": 98,
            "scam_type" : "lottery_fraud",
            "red_flags" : ["prize claim", "processing fee",
                           "Dubai Government impersonation",
                           "urgency deadline"]
        },
        "risks": {
            "technical_risk": "HIGH",
            "cultural_risk" : "VERY HIGH",
            "nlp_risk"      : "HIGH"
        }
    },
    {
        "id"     : "seed_004",
        "message": """ZaviyarHayat Group | WAS Group | FastInsu is hiring.
            Book your interview slot now — first come first served.
            marketing@zaviyarhayatgroup.com
            Limited slots available. No experience required.""",
        "verdict": {
            "is_scam"   : True,
            "confidence": 80,
            "scam_type" : "job_scam",
            "red_flags" : ["multiple company names",
                           "mass recruitment language",
                           "unverifiable company",
                           "domain 64 days old"]
        },
        "risks": {
            "technical_risk": "VERY HIGH",
            "cultural_risk" : "HIGH",
            "nlp_risk"      : "MEDIUM"
        }
    },
    {
        "id"     : "seed_005",
        "message": """Hi, your Noon order #UAE-2847361 has been shipped.
            Expected delivery tomorrow between 2-6pm.
            Track at https://noon.com/uae/track/2847361
            Noon Customer Service""",
        "verdict": {
            "is_scam"   : False,
            "confidence": 95,
            "scam_type" : "not_a_scam",
            "red_flags" : []
        },
        "risks": {
            "technical_risk": "LOW",
            "cultural_risk" : "LOW",
            "nlp_risk"      : "LOW"
        }
    },
    {
        "id"     : "seed_006",
        "message": """Dear candidate, FastInsu and WAS Group are conducting
            walk-in interviews. Book your slot on first come first served
            basis. Send CV to recruitment@wasgroup-hiring.com.
            Multiple positions available, no experience needed.""",
        "verdict": {
            "is_scam"   : True,
            "confidence": 75,
            "scam_type" : "job_scam",
            "red_flags" : ["mass recruitment", "unverifiable company",
                           "no requirements listed",
                           "suspicious email domain"]
        },
        "risks": {
            "technical_risk": "HIGH",
            "cultural_risk" : "HIGH",
            "nlp_risk"      : "MEDIUM"
        }
    },
    {
    "id"     : "seed_007",
    "message": """Dear Khan, I hope you are doing well. I am reaching
        out because we have open roles that might interest you.
        Please visit our website to see current openings and apply
        directly. Best regards, Karen Evans, Operations Manager""",
    "verdict": {
        "is_scam"   : False,
        "confidence": 90,
        "scam_type" : "not_a_scam",
        "red_flags" : []
    },
    "risks": {
        "technical_risk": "LOW",
        "cultural_risk" : "LOW",
        "nlp_risk"      : "LOW"
    }
},
{
    "id"     : "seed_008",
    "message": """Hi, I came across your profile and wanted to reach
        out about a position at our consulting firm. We have several
        openings listed on our LinkedIn page and company website.
        Feel free to apply if anything looks like a good fit.""",
    "verdict": {
        "is_scam"   : False,
        "confidence": 88,
        "scam_type" : "not_a_scam",
        "red_flags" : []
    },
    "risks": {
        "technical_risk": "LOW",
        "cultural_risk" : "LOW",
        "nlp_risk"      : "LOW"
    }
},
]

print("\nSeeding ChromaDB with known cases...")
for case in seed_cases:
    store_case(
        case_id        = case["id"],
        message        = case["message"],
        verdict        = case["verdict"],
        agents_findings= case["risks"]
    )

print(f"\n✅ Memory seeded with {len(seed_cases)} known cases")
print(f"   Total cases in ChromaDB: {collection.count()}")


Seeding ChromaDB with known cases...
   💾 Stored case seed_001 in memory
      Scam: True | Score: 80 | Type: job_scam
   💾 Stored case seed_002 in memory
      Scam: True | Score: 95 | Type: phishing
   💾 Stored case seed_003 in memory
      Scam: True | Score: 98 | Type: lottery_fraud
   💾 Stored case seed_004 in memory
      Scam: True | Score: 80 | Type: job_scam
   💾 Stored case seed_005 in memory
      Scam: False | Score: 95 | Type: not_a_scam
   💾 Stored case seed_006 in memory
      Scam: True | Score: 75 | Type: job_scam
   💾 Stored case seed_007 in memory
      Scam: False | Score: 90 | Type: not_a_scam
   💾 Stored case seed_008 in memory
      Scam: False | Score: 88 | Type: not_a_scam

✅ Memory seeded with 8 known cases
   Total cases in ChromaDB: 8


In [8]:
# CONCEPT: Before integrating with agents, verify that
# similarity search works correctly on its own.
# This is like unit testing your cosine similarity function.

print("\n" + "="*60)
print("  TESTING SIMILARITY RETRIEVAL")
print("="*60)

test_queries = [
    "Book your interview slot, first come first served, WAS Group hiring",
    "Your bank account is suspended, verify immediately",
    "Your package has been delivered successfully",
]

for query in test_queries:
    print(f"\nQuery: {query[:60]}...")
    similar = retrieve_similar(query, n_results=2)
    if similar:
        for case in similar:
            print(f"  Match: {case['case_id']} | "
                  f"Similarity: {case['similarity']:.0%} | "
                  f"Scam: {case['metadata']['is_scam']}")
    else:
        print("  No similar cases above threshold")



  TESTING SIMILARITY RETRIEVAL

Query: Book your interview slot, first come first served, WAS Group...
  Match: seed_006 | Similarity: 82% | Scam: True
  Match: seed_004 | Similarity: 72% | Scam: True

Query: Your bank account is suspended, verify immediately...
  Match: seed_002 | Similarity: 82% | Scam: True

Query: Your package has been delivered successfully...
  Match: seed_005 | Similarity: 72% | Scam: False


In [9]:
# CONCEPT: We extend the Phase 3 AgentState with memory fields.
# The state now carries retrieved cases and memory boost results.

class AgentState(TypedDict):
    original_message  : str
    orchestrator_plan : str
    entities_found    : dict
    nlp_findings      : dict
    technical_findings: dict
    cultural_findings : dict
    risk_score        : dict
    final_verdict     : dict
    agents_completed  : list
    error_log         : list
    # NEW Phase 4 memory fields
    similar_cases     : list    # retrieved from ChromaDB
    memory_context    : str     # formatted for LLM
    memory_boost      : dict    # score adjustment from memory


In [10]:
# CONCEPT: All Phase 3 tools carry forward unchanged.
# Memory is a new layer — it does not replace any existing tools.

import re as _re

def extract_entities_python(message: str) -> dict:
    emails  = _re.findall(
        r'[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}', message)
    urls    = _re.findall(r'https?://[^\s<>"{}|\\^`\[\]]+', message)
    from urllib.parse import urlparse
    email_domains = list(set([e.split("@")[1] for e in emails if "@" in e]))
    url_domains   = []
    for url in urls:
        try:
            d = urlparse(url).netloc
            if d: url_domains.append(d)
        except: pass
    all_domains = list(set(email_domains + url_domains))
    print(f"    Python extractor: emails={emails} urls={urls} domains={all_domains}")
    return {"emails": emails, "urls": urls, "domains": all_domains}


def whois_lookup(domain: str) -> dict:
    print(f"    🔍 WHOIS: {domain}")
    try:
        w        = whois.whois(domain)
        creation = w.creation_date
        if isinstance(creation, list): creation = creation[0]
        if creation:
            if creation.tzinfo is None:
                creation = creation.replace(tzinfo=timezone.utc)
            age_days = (datetime.now(timezone.utc) - creation).days
        else:
            age_days = None
        if age_days is None:       age_risk = "unknown"
        elif age_days < 30:        age_risk = "VERY HIGH"
        elif age_days < 180:       age_risk = "HIGH"
        elif age_days < 365:       age_risk = "MEDIUM"
        else:                      age_risk = "LOW"
        print(f"      Age: {age_days} days → {age_risk}")
        return {"domain": domain, "age_days": age_days,
                "age_risk": age_risk,
                "registrar": str(w.registrar) if w.registrar else "unknown",
                "country": str(w.country) if w.country else "unknown",
                "status": "success"}
    except Exception as e:
        return {"domain": domain, "status": "failed",
                "error": str(e), "age_risk": "unknown"}


def virustotal_scan(url: str) -> dict:
    print(f"    🔍 VirusTotal: {url[:60]}")
    headers = {"x-apikey": VT_KEY}
    try:
        r = requests.post("https://www.virustotal.com/api/v3/urls",
                          headers=headers, data={"url": url}, timeout=15)
        if r.status_code != 200:
            return {"url": url, "status": "failed"}
        analysis_id = r.json()["data"]["id"]
        time.sleep(5)
        r2    = requests.get(
            f"https://www.virustotal.com/api/v3/analyses/{analysis_id}",
            headers=headers, timeout=15)
        stats = r2.json()["data"]["attributes"]["stats"]
        mal   = stats.get("malicious", 0)
        total = sum(stats.values())
        if mal >= 5: vt_risk = "VERY HIGH"
        elif mal >= 2: vt_risk = "HIGH"
        elif mal >= 1: vt_risk = "MEDIUM"
        else:         vt_risk = "LOW"
        print(f"      Malicious: {mal}/{total} → {vt_risk}")
        return {"url": url, "malicious": mal, "total_engines": total,
                "vt_risk": vt_risk, "status": "success"}
    except Exception as e:
        return {"url": url, "status": "failed", "error": str(e)}


def scrape_website(url: str) -> dict:
    print(f"    🔍 Scraping: {url[:60]}")
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        r    = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(r.text, "html.parser")
        for tag in soup(["script","style","nav","footer"]):
            tag.decompose()
        text = " ".join(soup.get_text().split())[:2000].lower()
        flags = {
            "login_form"       : bool(soup.find("form")),
            "urgency_language" : any(w in text for w in [
                "urgent","expires","act now","immediately"]),
            "prize_language"   : any(w in text for w in [
                "congratulations","winner","prize"]),
            "financial_request": any(w in text for w in [
                "bank account","wire transfer","processing fee"]),
        }
        found = [k for k, v in flags.items() if v]
        print(f"      Flags: {found}")
        return {"url": url, "flags_found": found,
                "flags_count": len(found), "status": "success"}
    except requests.exceptions.ConnectionError:
        return {"url": url, "status": "unreachable",
                "note": "Website unreachable — strong scam signal"}
    except Exception as e:
        return {"url": url, "status": "failed", "error": str(e)}


def analyze_email_domain(email: str) -> dict:
    print(f"    🔍 Email: {email}")
    if "@" not in email:
        return {"email": email, "status": "invalid"}
    prefix, domain = email.split("@", 1)
    free_providers = ["gmail.com","yahoo.com","hotmail.com","outlook.com"]
    sus_prefixes   = ["hralert","hr-alert","noreply-hr","jobs-alert",
                      "marketing","alert-team","recruitment-alert"]
    is_free    = domain.lower() in free_providers
    sus_prefix = any(p in prefix.lower() for p in sus_prefixes)
    try:
        dns.resolver.resolve(domain, "MX")
        has_mx = True
    except: has_mx = False
    signals = []
    if is_free:    signals.append("free provider")
    if sus_prefix: signals.append(f"suspicious prefix: {prefix}")
    if not has_mx: signals.append("no MX records")
    risk = "HIGH" if len(signals)>=2 else "MEDIUM" if signals else "LOW"
    print(f"      Risk: {risk} | signals: {signals}")
    return {"email": email, "domain": domain, "prefix": prefix,
            "is_free_provider": is_free, "suspicious_prefix": sus_prefix,
            "has_mx_records": has_mx, "risk_signals": signals,
            "email_risk": risk, "status": "success"}


def check_domain_mismatch(sender_email: str, body_domains: list) -> dict:
    if "@" not in sender_email: return {"status": "invalid"}
    sender_domain = sender_email.split("@")[1].lower()
    body_domains  = [d.lower().strip() for d in body_domains]
    exact_match   = sender_domain in body_domains
    mismatches    = [d for d in body_domains if d != sender_domain]
    risk = ("HIGH — sender domain does not match body domains"
            if not exact_match and mismatches else "LOW")
    return {"sender_domain": sender_domain, "body_domains": body_domains,
            "exact_match": exact_match, "mismatches": mismatches,
            "mismatch_risk": risk, "status": "success"}


def check_company_existence(company_name: str, domain: str) -> dict:
    print(f"    🔍 Company existence: {company_name[:40]}")
    signals = {"domain_checked": domain, "company_name": company_name,
               "name_domain_match": False, "existence_signals": [],
               "absence_signals": [], "existence_risk": "LOW"}
    name_clean  = company_name.lower().replace(" ","").replace("-","")
    domain_root = domain.lower().split(".")[0].replace("-","")
    if domain_root in name_clean or name_clean in domain_root:
        signals["name_domain_match"] = True
        signals["existence_signals"].append("name matches domain")
    else:
        signals["absence_signals"].append("name does not match domain")
    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        r    = requests.get(f"https://{domain}", headers=headers, timeout=8)
        soup = BeautifulSoup(r.text, "html.parser")
        text = soup.get_text().lower()
        if any(w in text for w in ["about us","our team","contact"]):
            signals["existence_signals"].append("has business content")
        else:
            signals["absence_signals"].append("lacks business content")
        if any(w in text for w in ["careers","jobs","vacancies"]):
            signals["existence_signals"].append("has careers section")
        else:
            signals["absence_signals"].append("no careers section")
        name_words = [w for w in company_name.lower().split() if len(w)>4]
        if any(w in text for w in name_words):
            signals["existence_signals"].append("name found on site")
        else:
            signals["absence_signals"].append("name not found on site")
    except requests.exceptions.ConnectionError:
        signals["absence_signals"].append("website unreachable")
    except Exception as e:
        signals["absence_signals"].append(f"check failed: {str(e)[:40]}")

    website_unreachable = any("unreachable" in s
                               for s in signals["absence_signals"])
    absence_count = len(signals["absence_signals"])
    if website_unreachable and absence_count >= 2: signals["existence_risk"]="HIGH"
    elif website_unreachable or absence_count >= 3: signals["existence_risk"]="HIGH"
    elif absence_count >= 2: signals["existence_risk"] = "MEDIUM"

    print(f"      Existence risk: {signals['existence_risk']}")
    return signals

In [11]:
def call_llm(system_prompt, user_message, temperature=0.1):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role":"system","content":system_prompt},
                  {"role":"user","content":user_message}],
        temperature=temperature, max_tokens=1500)
    return response.choices[0].message.content.strip()

def parse_json_response(raw):
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"): raw = raw[4:]
    try: return json.loads(raw.strip())
    except: return {"error": "parse failed", "raw": raw}


In [12]:
# CONCEPT: This is the entirely new agent in Phase 4.
# It runs BEFORE all other agents — it is the first node.
# Its only job: retrieve similar past cases from ChromaDB
# and prepare memory context for the orchestrator to use.
# The orchestrator then uses this context as prior evidence.

def memory_retrieval_agent(state: AgentState) -> AgentState:
    """
    ROLE: Memory librarian. Runs first — before all other agents.
    Searches ChromaDB for similar past cases using cosine similarity.
    Formats retrieved cases as context for the orchestrator.

    CONCEPT: This is RAG — Retrieval Augmented Generation.
    Instead of the LLM relying purely on its training, it reads
    relevant past cases as context before making any decision.
    """
    print("\n" + "="*60)
    print("  AGENT 0: MEMORY RETRIEVAL")
    print("="*60)
    print(f"  Cases in memory: {collection.count()}")

    # Retrieve similar past cases
    similar_cases = retrieve_similar(
        message             = state["original_message"],
        n_results           = 3,
        similarity_threshold= 0.65
    )

    if similar_cases:
        print(f"\n  ✅ Found {len(similar_cases)} similar past case(s):")
        for case in similar_cases:
            meta = case["metadata"]
            print(f"     Case {case['case_id']} | "
                  f"Similarity: {case['similarity']:.0%} | "
                  f"Scam: {meta.get('is_scam')} | "
                  f"Score: {meta.get('confidence')}%")
    else:
        print("  No similar cases found above threshold")

    # Build formatted context for the LLM to read
    memory_context = build_memory_context(similar_cases)

    state["similar_cases"] = similar_cases
    state["memory_context"] = memory_context
    state["agents_completed"] = ["memory_retrieval"]

    return state

In [13]:
# CONCEPT: The orchestrator now receives memory context as
# additional input. All other agents remain unchanged.
# Memory only affects the orchestrator and the final scoring.

def orchestrator_agent(state: AgentState) -> AgentState:
    print("\n" + "="*60)
    print("  AGENT 1: ORCHESTRATOR (memory-aware)")
    print("="*60)

    python_entities = extract_entities_python(state["original_message"])

    # MEMORY INTEGRATION: Include past cases in the prompt
    memory_section = ""
    if state.get("memory_context") and \
       "No similar" not in state["memory_context"]:
        memory_section = f"""
## MEMORY FROM PAST CASES:
{state['memory_context']}
Use this historical evidence to inform your priority assessment.
If past similar cases were confirmed scams, treat this as HIGH priority.
"""

    system_prompt = f"""
    You are the lead fraud investigator with access to past case history.
    {memory_section}

    ## IDENTITY CONSISTENCY CHECK:
    1. Multiple company names in one sender identity → SUSPICIOUS
    2. Email domain not matching claimed company → MISMATCH
    3. "Book a slot" / "first come first served" → mass spam

    Return ONLY this JSON:
    {{
      "claimed_identity"  : "who claims to send this",
      "requested_action"  : "what they want recipient to do",
      "identity_flags"    : ["identity inconsistencies found"],
      "identity_risk"     : "HIGH or MEDIUM or LOW",
      "memory_influenced" : true or false,
      "investigation_plan": "one sentence",
      "priority"          : "HIGH or MEDIUM or LOW"
    }}
    """

    raw    = call_llm(system_prompt,
                      f"Analyze:\n\n{state['original_message']}")
    result = parse_json_response(raw)

    print(f"  Identity flags   : {result.get('identity_flags', [])}")
    print(f"  Identity risk    : {result.get('identity_risk')}")
    print(f"  Memory influenced: {result.get('memory_influenced')}")
    print(f"  Priority         : {result.get('priority')}")

    state["entities_found"] = {
        "emails"          : python_entities["emails"],
        "urls"            : python_entities["urls"],
        "domains"         : python_entities["domains"],
        "claimed_identity": result.get("claimed_identity", ""),
        "requested_action": result.get("requested_action", ""),
        "identity_flags"  : result.get("identity_flags", []),
        "identity_risk"   : result.get("identity_risk", "LOW"),
    }
    state["orchestrator_plan"] = result.get("investigation_plan", "")
    state["agents_completed"].append("orchestrator")
    return state


def nlp_specialist_agent(state: AgentState) -> AgentState:
    print("\n" + "="*60)
    print("  AGENT 2: NLP SPECIALIST")
    print("="*60)
    system_prompt = """
    Analyze ONLY the text — not URLs or domains. Look for:
    1. Urgency language, 2. Threats, 3. Authority impersonation,
    4. Reward claims, 5. Info harvesting, 6. Generic greetings,
    7. Grammar issues, 8. Mass recruitment language
    ("book a slot", "first come first served", "limited slots")

    Return ONLY this JSON:
    {
      "urgency_found": true/false, "threat_found": true/false,
      "authority_claim": true/false, "reward_claim": true/false,
      "info_harvesting": true/false, "generic_greeting": true/false,
      "grammar_issues": true/false, "mass_recruitment": true/false,
      "key_phrases": ["phrases"], "sentiment": "aggressive/friendly/neutral/urgent",
      "manipulation_score": 0-100, "nlp_risk": "HIGH/MEDIUM/LOW",
      "nlp_summary": "one sentence"
    }
    """
    raw    = call_llm(system_prompt, f"Analyze:\n\n{state['original_message']}")
    result = parse_json_response(raw)

    # Rule-based risk override — do not trust LLM to self-assess
    signals = sum([bool(result.get("urgency_found")),
                   bool(result.get("threat_found")),
                   bool(result.get("authority_claim")),
                   bool(result.get("reward_claim")),
                   bool(result.get("info_harvesting")),
                   bool(result.get("mass_recruitment")),
                   bool(result.get("generic_greeting"))])
    if signals >= 4:   result["nlp_risk"] = "HIGH"
    elif signals >= 2: result["nlp_risk"] = "MEDIUM"
    else:              result["nlp_risk"] = "LOW"
    result["high_nlp_signals"] = signals

    print(f"  NLP signals: {signals} → risk: {result['nlp_risk']}")
    print(f"  Key phrases: {result.get('key_phrases', [])}")

    state["nlp_findings"] = result
    state["agents_completed"].append("nlp_specialist")
    return state


def technical_specialist_agent(state: AgentState) -> AgentState:
    print("\n" + "="*60)
    print("  AGENT 3: TECHNICAL SPECIALIST")
    print("="*60)
    entities = state.get("entities_found", {})
    evidence = {"whois_results": [], "virustotal": [],
                "email_analysis": [], "website_scrapes": [],
                "company_existence": {}, "domain_mismatch": {},
                "technical_risk": "LOW"}

    domains       = entities.get("domains", [])
    emails        = entities.get("emails",  [])
    urls          = entities.get("urls",    [])
    email_domains = [e.split("@")[1] for e in emails if "@" in e]
    all_domains   = list(set(domains + email_domains))

    for domain in all_domains:
        evidence["whois_results"].append(whois_lookup(domain))
        time.sleep(1)
    for email in emails:
        evidence["email_analysis"].append(analyze_email_domain(email))
    if emails and len(all_domains) > 1:
        evidence["domain_mismatch"] = check_domain_mismatch(
            emails[0], [d for d in all_domains if d not in emails[0]])
    claimed = entities.get("claimed_identity", "")
    if claimed and all_domains:
        evidence["company_existence"] = check_company_existence(
            claimed, all_domains[0])
    for url in urls[:2]:
        evidence["virustotal"].append(virustotal_scan(url))
        time.sleep(15)
    for url in urls[:1]:
        evidence["website_scrapes"].append(scrape_website(url))

    high_signals = 0
    for w in evidence["whois_results"]:
        if w.get("age_risk") in ["VERY HIGH","HIGH"]: high_signals += 1
    for e in evidence["email_analysis"]:
        if e.get("email_risk") == "HIGH": high_signals += 1
    for v in evidence["virustotal"]:
        if v.get("vt_risk") in ["VERY HIGH","HIGH"]: high_signals += 2
    for s in evidence["website_scrapes"]:
        if s.get("flags_count",0) >= 2: high_signals += 1
    company_risk = evidence["company_existence"].get("existence_risk","LOW")
    if company_risk == "HIGH":   high_signals += 2
    elif company_risk == "MEDIUM": high_signals += 1
    if "HIGH" in str(evidence["domain_mismatch"].get("mismatch_risk","")):
        high_signals += 1
    if entities.get("identity_risk") == "HIGH": high_signals += 1

    nlp_risk      = state.get("nlp_findings",{}).get("nlp_risk","LOW")
    cultural_risk = state.get("cultural_findings",{}).get("cultural_risk","LOW")
    if (not urls and nlp_risk in ["HIGH","VERY HIGH"] and
            cultural_risk in ["HIGH","VERY HIGH"]):
        high_signals += 2
        print("  ⚠ Compensating control: no URLs + high NLP/cultural")

    if high_signals >= 4:   evidence["technical_risk"] = "VERY HIGH"
    elif high_signals >= 3: evidence["technical_risk"] = "HIGH"
    elif high_signals >= 2: evidence["technical_risk"] = "MEDIUM"
    elif high_signals >= 1: evidence["technical_risk"] = "LOW-MEDIUM"
    else:                   evidence["technical_risk"] = "LOW"

    print(f"  High signals: {high_signals} → {evidence['technical_risk']}")
    state["technical_findings"] = evidence
    state["agents_completed"].append("technical_specialist")
    return state


def cultural_context_agent(state: AgentState) -> AgentState:
    print("\n" + "="*60)
    print("  AGENT 4: CULTURAL CONTEXT")
    print("="*60)
    system_prompt = """
    UAE fraud specialist. Detect UAE-specific patterns:

    JOB SCAMS: Fake ADNOC/Emirates/Etisalat hiring, unrealistic salaries
    (AED 15,000+), upfront fees, unverifiable companies

    UNSOLICITED RECRUITMENT (3+ flags = HIGH risk):
    - Unsolicited contact never applied to
    - Multiple company names in one sender
    - "Book a slot", "first come first served", "limited slots"
    - No job requirements or qualifications mentioned
    - No reference to where they found your CV
    - Insurance/finance recruiting for unrelated roles

    FINANCIAL SCAMS: Dubai Government draws, UAE Central Bank alerts
    IMPERSONATION: UAE Police, embassies, major banks

    Return ONLY this JSON:
    {
      "uae_entity_impersonated": "name or null",
      "known_uae_scam_pattern": true/false,
      "unrealistic_offer": true/false,
      "unsolicited_recruitment": true/false,
      "unsolicited_flags_found": ["flags"],
      "multiple_company_names": true/false,
      "mass_recruitment_language": true/false,
      "cultural_red_flags": ["flags"],
      "cultural_risk": "HIGH or MEDIUM or LOW",
      "cultural_summary": "one sentence"
    }
    """
    raw    = call_llm(system_prompt,
                      f"Analyze:\n\n{state['original_message']}")
    result = parse_json_response(raw)
    print(f"  Cultural risk  : {result.get('cultural_risk')}")
    print(f"  Cultural flags : {result.get('cultural_red_flags',[])}")
    state["cultural_findings"] = result
    state["agents_completed"].append("cultural_context")
    return state


def risk_scorer_agent(state: AgentState) -> AgentState:
    """
    Phase 4 risk scorer: calculates base score then applies
    memory boost from similar past cases.
    """
    print("\n" + "="*60)
    print("  AGENT 5: RISK SCORER (memory-boosted)")
    print("="*60)

    def risk_to_score(risk):
        return {"VERY HIGH":95,"HIGH":75,"LOW-MEDIUM":55,
                "MEDIUM":50,"LOW":15,"unknown":30}.get(risk, 30)

    tech_risk     = state.get("technical_findings",{}).get("technical_risk","LOW")
    cultural_risk = state.get("cultural_findings", {}).get("cultural_risk", "LOW")
    nlp_risk      = state.get("nlp_findings",      {}).get("nlp_risk",      "LOW")

    tech_score     = risk_to_score(tech_risk)
    cultural_score = risk_to_score(cultural_risk)
    nlp_score      = risk_to_score(nlp_risk)

    # Adaptive weights
    has_technical = bool(
        state.get("technical_findings",{}).get("whois_results") or
        state.get("technical_findings",{}).get("email_analysis"))
    if has_technical:
        weights = {"technical":0.50,"cultural":0.30,"nlp":0.20}
        print("  Standard weights (technical evidence available)")
    else:
        weights = {"technical":0.20,"cultural":0.50,"nlp":0.30}
        print("  ⚠ Adaptive weights (no technical evidence)")

    base_score = int(
        tech_score     * weights["technical"] +
        cultural_score * weights["cultural"]  +
        nlp_score      * weights["nlp"]
    )

    print(f"\n  Base calculation:")
    print(f"  Technical : {tech_score} × {int(weights['technical']*100)}% = "
          f"{int(tech_score*weights['technical'])}")
    print(f"  Cultural  : {cultural_score} × {int(weights['cultural']*100)}% = "
          f"{int(cultural_score*weights['cultural'])}")
    print(f"  NLP       : {nlp_score} × {int(weights['nlp']*100)}% = "
          f"{int(nlp_score*weights['nlp'])}")
    print(f"  Base score: {base_score}/100")

    # MEMORY BOOST — apply adjustment from similar past cases
    similar_cases = state.get("similar_cases", [])
    memory_boost  = calculate_memory_boost(similar_cases, base_score)
    final_score   = memory_boost["boosted_score"]

    if memory_boost["boost_applied"] != 0:
        print(f"\n  Memory boost: {memory_boost['boost_applied']:+d} points")
        for reason in memory_boost["boost_reason"]:
            print(f"     • {reason}")
        print(f"  Final score : {final_score}/100")
    else:
        print(f"  No memory boost applied")
        print(f"  Final score : {final_score}/100")

    if final_score >= 85:   confidence = "VERY HIGH"
    elif final_score >= 65: confidence = "HIGH"
    elif final_score >= 45: confidence = "MEDIUM"
    else:                   confidence = "LOW"

    state["risk_score"] = {
        "base_score"    : base_score,
        "final_score"   : final_score,
        "confidence"    : confidence,
        "is_scam"       : final_score >= 55,
        "breakdown"     : {
            "technical_score" : tech_score,
            "cultural_score"  : cultural_score,
            "nlp_score"       : nlp_score,
        },
        "individual_risks": {
            "technical_risk": tech_risk,
            "cultural_risk" : cultural_risk,
            "nlp_risk"      : nlp_risk,
        }
    }
    state["memory_boost"] = memory_boost
    state["agents_completed"].append("risk_scorer")
    return state


def verdict_agent(state: AgentState) -> AgentState:
    print("\n" + "="*60)
    print("  AGENT 6: FINAL VERDICT")
    print("="*60)

    all_findings = {
        "original_message"  : state["original_message"][:300],
        "entities_found"    : state.get("entities_found",     {}),
        "nlp_findings"      : state.get("nlp_findings",       {}),
        "technical_findings": state.get("technical_findings", {}),
        "cultural_findings" : state.get("cultural_findings",  {}),
        "risk_score"        : state.get("risk_score",         {}),
        "similar_past_cases": len(state.get("similar_cases",  [])),
        "memory_boost"      : state.get("memory_boost",       {}),
    }

    system_prompt = """
    Lead fraud investigator writing final case report.
    Consider both agent findings AND memory from similar past cases.

    Return ONLY this JSON:
    {
      "is_scam"           : true/false,
      "confidence"        : 0-100,
      "scam_type"         : "phishing/lottery_fraud/advance_fee/romance_scam/
                             investment_fraud/impersonation/job_scam/
                             tech_support_scam/not_a_scam/unknown",
      "verdict_summary"   : "2-3 sentence plain English verdict",
      "key_evidence"      : ["top 3-5 evidence pieces"],
      "red_flags"         : ["all red flags"],
      "safe_indicators"   : ["any safe signals"],
      "recommended_action": "clear action for recipient",
      "memory_note"       : "how past cases influenced this verdict or null"
    }
    """

    raw    = call_llm(system_prompt, json.dumps(all_findings, default=str))
    result = parse_json_response(raw)

    if state.get("risk_score"):
        result["confidence"] = state["risk_score"]["final_score"]
        result["is_scam"]    = state["risk_score"]["is_scam"]

    state["final_verdict"] = result
    state["agents_completed"].append("verdict")
    return state

In [14]:
# CONCEPT: Memory retrieval agent added as first node.
# Everything else stays the same — memory is a new layer,
# not a replacement of any existing agent.

def build_graph():
    graph = StateGraph(AgentState)

    # All nodes — memory_retrieval is new, runs first
    graph.add_node("memory_retrieval", memory_retrieval_agent)
    graph.add_node("orchestrator",     orchestrator_agent)
    graph.add_node("nlp",              nlp_specialist_agent)
    graph.add_node("technical",        technical_specialist_agent)
    graph.add_node("cultural",         cultural_context_agent)
    graph.add_node("risk_scorer",      risk_scorer_agent)
    graph.add_node("verdict",          verdict_agent)

    # Flow — memory retrieval runs before orchestrator
    graph.set_entry_point("memory_retrieval")
    graph.add_edge("memory_retrieval", "orchestrator")
    graph.add_edge("orchestrator",     "nlp")
    graph.add_edge("nlp",              "technical")
    graph.add_edge("technical",        "cultural")
    graph.add_edge("cultural",         "risk_scorer")
    graph.add_edge("risk_scorer",      "verdict")
    graph.add_edge("verdict",          END)

    return graph.compile()


investigation_graph = build_graph()
print("✅ Phase 4 graph compiled")
print("   Flow: memory → orchestrator → nlp → technical → cultural → scorer → verdict")


✅ Phase 4 graph compiled
   Flow: memory → orchestrator → nlp → technical → cultural → scorer → verdict


In [15]:
def display_verdict(state: AgentState):
    verdict     = state.get("final_verdict", {})
    risk_score  = state.get("risk_score",    {})
    memory_boost= state.get("memory_boost",  {})
    is_scam     = verdict.get("is_scam", False)
    confidence  = verdict.get("confidence", 0)
    filled      = int(confidence / 10)
    bar         = "█" * filled + "░" * (10 - filled)

    print(f"\n{'─'*60}")
    print(f"  VERDICT   : {'🚨 SCAM DETECTED' if is_scam else '✅ LIKELY SAFE'}")
    print(f"  Score     : [{bar}] {confidence}/100")
    print(f"  Base score: {risk_score.get('base_score', confidence)}/100")
    print(f"  Memory boost: {memory_boost.get('boost_applied', 0):+d} points")
    print(f"  Scam type : {verdict.get('scam_type','').replace('_',' ').title()}")
    print(f"  Agents    : {len(state.get('agents_completed',[]))} completed")
    print(f"{'─'*60}")

    similar = state.get("similar_cases", [])
    if similar:
        print(f"\n  🧠 MEMORY ({len(similar)} similar past case(s)):")
        for c in similar:
            meta = c["metadata"]
            print(f"     Case {c['case_id']} | "
                  f"Similarity: {c['similarity']:.0%} | "
                  f"{'SCAM' if meta['is_scam']=='True' else 'SAFE'} | "
                  f"Score: {meta['confidence']}%")

    if memory_boost.get("boost_reason"):
        print(f"\n  📈 MEMORY BOOST REASONS:")
        for r in memory_boost["boost_reason"]:
            print(f"     • {r}")

    key_evidence = verdict.get("key_evidence", [])
    if key_evidence:
        print(f"\n  🔬 KEY EVIDENCE:")
        for e in key_evidence: print(f"     • {e}")

    red_flags = verdict.get("red_flags", [])
    if red_flags:
        print(f"\n  🚩 RED FLAGS:")
        for f in red_flags: print(f"     • {f}")

    print(f"\n  📋 SUMMARY:")
    print(f"     {verdict.get('verdict_summary','')}")

    if verdict.get("memory_note"):
        print(f"\n  🧠 MEMORY NOTE:")
        print(f"     {verdict['memory_note']}")

    print(f"\n  💡 ACTION:")
    print(f"     {verdict.get('recommended_action','')}")
    print(f"{'─'*60}\n")


def investigate(message: str) -> AgentState:
    initial_state: AgentState = {
        "original_message"  : message,
        "orchestrator_plan" : "",
        "entities_found"    : {},
        "nlp_findings"      : {},
        "technical_findings": {},
        "cultural_findings" : {},
        "risk_score"        : {},
        "final_verdict"     : {},
        "agents_completed"  : [],
        "error_log"         : [],
        "similar_cases"     : [],
        "memory_context"    : "",
        "memory_boost"      : {},
    }
    print("\n" + "="*60)
    print("  PHASE 4: MEMORY-AUGMENTED INVESTIGATION")
    print(f"  {collection.count()} cases in memory")
    print("="*60)
    return investigation_graph.invoke(initial_state)


def investigate_and_store(message: str,
                          case_id: str = None) -> AgentState:
    """
    Investigates a message AND stores the result in memory.
    Use this instead of investigate() to grow the memory
    database with every new case.
    """
    if case_id is None:
        case_id = f"case_{int(time.time())}"

    # Run full investigation
    final_state = investigate(message)

    # Store result in ChromaDB memory
    verdict = final_state.get("final_verdict", {})
    tech    = final_state.get("technical_findings", {})
    cultural= final_state.get("cultural_findings",  {})
    nlp     = final_state.get("nlp_findings",       {})

    store_case(
        case_id        = case_id,
        message        = message,
        verdict        = verdict,
        agents_findings= {
            "technical_risk": tech.get("technical_risk", "unknown"),
            "cultural_risk" : cultural.get("cultural_risk", "unknown"),
            "nlp_risk"      : nlp.get("nlp_risk", "unknown"),
        }
    )

    print(f"\n  💾 Case stored in memory as: {case_id}")
    print(f"  📊 Total cases in memory   : {collection.count()}")

    return final_state


In [16]:
test_messages = [

    # TEST 1: Very similar to seed_004 — should get memory boost
    # seed_004 was the ZaviyarHayat group email — 80% scam
    # This similar variant should score higher due to memory
    """
    From: recruitment@wasgroup-uae.com
    WAS Group | ZaviyarHayat | FastInsu — NOW HIRING
    Book your interview slot immediately.
    First come first served — limited slots available.
    No experience required. Multiple positions open.
    """,

    # TEST 2: Similar to seed_002 bank phishing — memory should boost
    """
    Your FAB account has been suspended due to unusual activity.
    Verify your account now: http://fab-secure-verify.tk/login
    Account will be permanently closed within 2 hours.
    FAB Security Team
    """,

    # TEST 3: Legitimate — memory should not boost, safe cases match
    """
    Hi, your Amazon.ae order #AE-3921847 is out for delivery.
    Expected today between 3-7pm.
    Track: https://amazon.ae/track/3921847
    """,
]

results = []
for i, message in enumerate(test_messages, 1):
    print(f"\n{'='*60}")
    print(f"  TEST {i} of {len(test_messages)}")
    print(f"{'='*60}")
    # Use investigate_and_store so memory grows with each test
    final_state = investigate_and_store(message, f"test_{i:03d}")
    display_verdict(final_state)
    results.append(final_state)
    time.sleep(2)



  TEST 1 of 3

  PHASE 4: MEMORY-AUGMENTED INVESTIGATION
  8 cases in memory

  AGENT 0: MEMORY RETRIEVAL
  Cases in memory: 8

  ✅ Found 3 similar past case(s):
     Case seed_004 | Similarity: 89% | Scam: True | Score: 80%
     Case seed_006 | Similarity: 87% | Scam: True | Score: 75%
     Case seed_001 | Similarity: 74% | Scam: True | Score: 80%

  AGENT 1: ORCHESTRATOR (memory-aware)
    Python extractor: emails=['recruitment@wasgroup-uae.com'] urls=[] domains=['wasgroup-uae.com']
  Identity flags   : ['Multiple company names in one sender identity', 'Email domain not matching claimed company', 'Mass recruitment language']
  Identity risk    : HIGH
  Memory influenced: True
  Priority         : HIGH

  AGENT 2: NLP SPECIALIST
  NLP signals: 2 → risk: MEDIUM
  Key phrases: ['Book your interview slot immediately', 'First come first served', 'limited slots available']

  AGENT 3: TECHNICAL SPECIALIST
    🔍 WHOIS: wasgroup-uae.com
    🔍 Email: recruitment@wasgroup-uae.com
      Risk: 

ERROR:whois.whois:Error trying to connect to socket: closing socket - timed out


      Age: None days → unknown
    🔍 Company existence: FAB Security Team
      Existence risk: HIGH
    🔍 VirusTotal: http://fab-secure-verify.tk/login
      Malicious: 0/0 → LOW
    🔍 Scraping: http://fab-secure-verify.tk/login
  High signals: 3 → HIGH

  AGENT 4: CULTURAL CONTEXT
  Cultural risk  : HIGH
  Cultural flags : ['urgency', 'suspicious link']

  AGENT 5: RISK SCORER (memory-boosted)
  Standard weights (technical evidence available)

  Base calculation:
  Technical : 75 × 50% = 37
  Cultural  : 75 × 30% = 22
  NLP       : 75 × 20% = 15
  Base score: 75/100

  Memory boost: +5 points
     • Similar past scam found (similarity: 81%)
  Final score : 80/100

  AGENT 6: FINAL VERDICT
   💾 Stored case test_002 in memory
      Scam: True | Score: 80 | Type: phishing

  💾 Case stored in memory as: test_002
  📊 Total cases in memory   : 10

────────────────────────────────────────────────────────────
  VERDICT   : 🚨 SCAM DETECTED
  Score     : [████████░░] 80/100
  Base score: 75/10

In [17]:
# CONCEPT: Run the same scam type again and watch memory boost
# increase because we now have MORE similar cases stored.

print("\n" + "="*60)
print("  MEMORY GROWTH DEMONSTRATION")
print("  Running a new variant after storing test cases...")
print("="*60)

new_variant = """
From: hr@wasgroupuae-hiring.net
Urgent Hiring — WAS Group and Associated Companies
Walk-in interviews this week. Book your slot now.
First come first served. Bring CV and Emirates ID.
All nationalities welcome. No prior experience needed.
"""

final = investigate_and_store(new_variant, "variant_001")
display_verdict(final)

print(f"\n  Total cases now in memory: {collection.count()}")
print("  Every future similar scam will score higher.")



  MEMORY GROWTH DEMONSTRATION
  Running a new variant after storing test cases...

  PHASE 4: MEMORY-AUGMENTED INVESTIGATION
  11 cases in memory

  AGENT 0: MEMORY RETRIEVAL
  Cases in memory: 11

  ✅ Found 3 similar past case(s):
     Case test_001 | Similarity: 88% | Scam: True | Score: 85%
     Case seed_006 | Similarity: 81% | Scam: True | Score: 75%
     Case seed_001 | Similarity: 79% | Scam: True | Score: 80%

  AGENT 1: ORCHESTRATOR (memory-aware)
    Python extractor: emails=['hr@wasgroupuae-hiring.net'] urls=[] domains=['wasgroupuae-hiring.net']
  Identity flags   : ['Multiple company names in one sender', 'Email domain not matching claimed company']
  Identity risk    : HIGH
  Memory influenced: True
  Priority         : HIGH

  AGENT 2: NLP SPECIALIST
  NLP signals: 3 → risk: MEDIUM
  Key phrases: ['Urgent Hiring', 'Book your slot now', 'First come first served']

  AGENT 3: TECHNICAL SPECIALIST
    🔍 WHOIS: wasgroupuae-hiring.net
    🔍 Email: hr@wasgroupuae-hiring.net
  

In [18]:
print("\n--- YOUR TURN ---")
your_message = """
ZaviyarhayatGroup <marketing@zaviyarhayatgroup.com>
Wed, Apr 22, 5:57 PM (9 days ago)
to me

Hi Kim,

I’m sending over the updated list of locations and times for the upcoming interview sessions in Dubai and the wider GCC.
We have refreshed the schedule with direct contact details for the hiring teams. You can find the specific venue addresses and times for this week at the link below:

BOOK YOUR SLOT HERE

Quick notes for this week:
1.      Most sessions are first-come, first-served due to venue capacity.
2.      Direct employer contacts are included in the listings.
3.      There are no costs or fees to attend these sessions.

I hope this helps with your search.

Best,
Recruitment Team WAS Group | FastInsu
"""
your_state = investigate_and_store(your_message, "user_test_001")
display_verdict(your_state)


--- YOUR TURN ---

  PHASE 4: MEMORY-AUGMENTED INVESTIGATION
  12 cases in memory

  AGENT 0: MEMORY RETRIEVAL
  Cases in memory: 12

  ✅ Found 3 similar past case(s):
     Case test_001 | Similarity: 84% | Scam: True | Score: 85%
     Case variant_001 | Similarity: 83% | Scam: True | Score: 85%
     Case seed_006 | Similarity: 78% | Scam: True | Score: 75%

  AGENT 1: ORCHESTRATOR (memory-aware)
    Python extractor: emails=['marketing@zaviyarhayatgroup.com'] urls=[] domains=['zaviyarhayatgroup.com']
  Identity flags   : ['Multiple company names in one sender', 'Email domain not matching claimed company']
  Identity risk    : HIGH
  Memory influenced: True
  Priority         : HIGH

  AGENT 2: NLP SPECIALIST
  NLP signals: 3 → risk: MEDIUM
  Key phrases: ['BOOK YOUR SLOT HERE', 'first-come, first-served']

  AGENT 3: TECHNICAL SPECIALIST
    🔍 WHOIS: zaviyarhayatgroup.com
      Age: 67 days → HIGH
    🔍 Email: marketing@zaviyarhayatgroup.com
      Risk: MEDIUM | signals: ['suspicious

In [19]:
print("\n--- YOUR TURN ---")
your_message = """
karen@faze3consulting.com
1:22 PM (4 hours ago)
to me

Dear Khan,

I hope you're doing well. I'm reaching out because we currently have a number of open roles that might be of interest to you.

Rather than highlight just one, I'd encourage you to take a look at the full list — you can find all our current openings on our website (https://faze3consulting.com/careers) and on our LinkedIn page (https://www.linkedin.com/company/faze-3-consulting/). Each listing includes the full job description, requirements, and instructions on how to apply.

If you see something that looks like a good fit, please feel free to apply directly through the link.

Best regards,

---------------------
Karen Evans
Operations Manager
Faze 3 Consulting
"""
your_state = investigate_and_store(your_message, "user_test_001")
display_verdict(your_state)


--- YOUR TURN ---

  PHASE 4: MEMORY-AUGMENTED INVESTIGATION
  13 cases in memory

  AGENT 0: MEMORY RETRIEVAL
  Cases in memory: 13

  ✅ Found 3 similar past case(s):
     Case seed_008 | Similarity: 83% | Scam: False | Score: 88%
     Case seed_007 | Similarity: 80% | Scam: False | Score: 90%
     Case user_test_001 | Similarity: 70% | Scam: True | Score: 95%

  AGENT 1: ORCHESTRATOR (memory-aware)
    Python extractor: emails=['karen@faze3consulting.com'] urls=['https://faze3consulting.com/careers)', 'https://www.linkedin.com/company/faze-3-consulting/).'] domains=['www.linkedin.com', 'faze3consulting.com']
  Identity flags   : []
  Identity risk    : LOW
  Memory influenced: True
  Priority         : LOW

  AGENT 2: NLP SPECIALIST
  NLP signals: 1 → risk: LOW
  Key phrases: ['open roles', 'current openings']

  AGENT 3: TECHNICAL SPECIALIST
    🔍 WHOIS: www.linkedin.com
      Age: 8584 days → LOW
    🔍 WHOIS: faze3consulting.com


ERROR:whois.whois:Error trying to connect to socket: closing socket - timed out


      Age: 1789 days → LOW
    🔍 Email: karen@faze3consulting.com
      Risk: LOW | signals: []
    🔍 Company existence: Karen Evans, Operations Manager at Faze 
      Existence risk: MEDIUM
    🔍 VirusTotal: https://faze3consulting.com/careers)
      Malicious: 0/0 → LOW
    🔍 VirusTotal: https://www.linkedin.com/company/faze-3-consulting/).
      Malicious: 0/0 → LOW
    🔍 Scraping: https://faze3consulting.com/careers)
      Flags: []
  High signals: 2 → MEDIUM

  AGENT 4: CULTURAL CONTEXT
  Cultural risk  : LOW
  Cultural flags : []

  AGENT 5: RISK SCORER (memory-boosted)
  Standard weights (technical evidence available)

  Base calculation:
  Technical : 50 × 50% = 25
  Cultural  : 15 × 30% = 4
  NLP       : 15 × 20% = 3
  Base score: 32/100
  No memory boost applied
  Final score : 32/100

  AGENT 6: FINAL VERDICT
   💾 Stored case user_test_001 in memory
      Scam: False | Score: 32 | Type: not_a_scam

  💾 Case stored in memory as: user_test_001
  📊 Total cases in memory   : 13


In [20]:
print("\n" + "="*60)
print("  MEMORY DATABASE STATISTICS")
print("="*60)

all_cases = collection.get(include=["metadatas"])
metadatas = all_cases["metadatas"]

total     = len(metadatas)
scams     = sum(1 for m in metadatas if m.get("is_scam") == "True")
safe      = sum(1 for m in metadatas if m.get("is_scam") == "False")
avg_score = (sum(int(m.get("confidence",0)) for m in metadatas) / total
             if total > 0 else 0)

scam_types = {}
for m in metadatas:
    st = m.get("scam_type", "unknown")
    scam_types[st] = scam_types.get(st, 0) + 1

print(f"  Total cases stored : {total}")
print(f"  Confirmed scams    : {scams}")
print(f"  Confirmed safe     : {safe}")
print(f"  Average confidence : {avg_score:.1f}%")
print(f"\n  Cases by scam type:")
for st, count in sorted(scam_types.items(),
                          key=lambda x: x[1], reverse=True):
    print(f"     {st:25}: {count}")



  MEMORY DATABASE STATISTICS
  Total cases stored : 13
  Confirmed scams    : 8
  Confirmed safe     : 5
  Average confidence : 78.8%

  Cases by scam type:
     job_scam                 : 5
     not_a_scam               : 5
     phishing                 : 2
     lottery_fraud            : 1
